In [ ]:
"""
Diabetes diagnosis with machine learning
Comparison of six supervised classifiers on the Pima Indians Diabetes dataset.

Author: Miguel Castro

Each model is evaluated on accuracy, precision, and recall on a held-out test set.
Recall is treated as the priority metric: in a screening context, a false negative
(a missed case) is the costly error.
"""

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 123
SPLIT_SEED = 12345


def load_data(path="patients_diabetis.txt"):
    """Read the tab-separated dataset and split into predictors and label."""
    df = pd.read_csv(path, sep="\t")
    X = df.drop("class", axis=1)
    y = df["class"]
    return df, X, y


def make_split(X, y):
    """Standardise the features and make a stratified 67/33 train/test split."""
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    return train_test_split(
        X_scaled, y, test_size=0.33, random_state=SPLIT_SEED, stratify=y
    )


def evaluate(model, X_train, X_test, y_train, y_test):
    """Fit a model and return accuracy, precision, and recall on the test set."""
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
    }


def main():
    df, X, y = load_data()
    print("Class distribution:")
    print(y.value_counts(), "\n")

    X_train, X_test, y_train, y_test = make_split(X, y)

    def ev(model):
        return evaluate(model, X_train, X_test, y_train, y_test)

    results = []

    # k-NN with several values of k
    for k in [1, 3, 5, 7, 11]:
        res = ev(KNeighborsClassifier(n_neighbors=k))
        res["Model"] = f"k-NN (k={k})"
        results.append(res)

    # Naive Bayes
    res = ev(GaussianNB())
    res["Model"] = "Naive Bayes"
    results.append(res)

    # Neural network (MLP) with three architectures
    for layers in [(12, 6), (20, 10), (20, 10, 5)]:
        res = ev(MLPClassifier(hidden_layer_sizes=layers, max_iter=1000,
                               random_state=RANDOM_STATE))
        res["Model"] = f"MLP {layers}"
        results.append(res)

    # SVM with linear and RBF kernels
    for kernel in ["linear", "rbf"]:
        res = ev(SVC(kernel=kernel, random_state=RANDOM_STATE))
        res["Model"] = f"SVM ({kernel})"
        results.append(res)

    # Decision tree
    res = ev(DecisionTreeClassifier(random_state=RANDOM_STATE))
    res["Model"] = "Decision tree"
    results.append(res)

    # Random forest with 100 and 200 trees
    for n in [100, 200]:
        res = ev(RandomForestClassifier(n_estimators=n, random_state=RANDOM_STATE))
        res["Model"] = f"Random forest ({n} trees)"
        results.append(res)

    # Summary table sorted by accuracy
    summary = (
        pd.DataFrame(results)[["Model", "Accuracy", "Precision", "Recall"]]
        .sort_values("Accuracy", ascending=False)
        .reset_index(drop=True)
    )
    pd.set_option("display.float_format", lambda v: f"{v:.3f}")
    print("Model comparison (sorted by accuracy):")
    print(summary.to_string(index=False))

    best = summary.iloc[0]
    print(f"\nBest overall balance: {best['Model']} "
          f"(accuracy {best['Accuracy']:.3f}, recall {best['Recall']:.3f}).")


if __name__ == "__main__":
    main()
